# Digital Wellbeing Coach
## Smartphone Addiction Risk Predictor — Model Notebook
**Reine Mizero | BSc Software Engineering | African Leadership University**

---

### Dataset
**Source:** Smartphone Addiction and Cyberbullying in Early Adolescents (Indonesia, N=394)  
**Instrument:** Smartphone Addiction Scale – Short Version (SAS-SV), Kwon et al. (2013)  
**Target variable:** `RESULT` — binary addiction label (1 = addicted, 0 = not addicted)  
**Cut-offs applied:** Female (P) ≥ 33, Male (L) ≥ 31

### Domain Shift Acknowledgement
The training dataset was collected from early adolescents aged 12–16 in Indonesian secondary schools. The target deployment population is university students aged 18–25 in Kigali, Rwanda. This age group difference is contextually justified: the Government of Rwanda has implemented a national ban on smartphone use in primary and secondary schools (Rwanda Ministry of Education, 2018), meaning intensive smartphone use among Rwandan youth occurs primarily during university years when students have unrestricted device access for the first time.

### Notebook Structure
1. Environment Setup & Data Loading  
2. Exploratory Data Analysis (EDA) & Visualizations  
3. Data Preprocessing & Feature Engineering  
4. Model Architecture & Training (LR, RF, XGBoost)  
5. Performance Metrics & Model Comparison  
6. SHAP Explainability Analysis  
7. Summary & Next Steps


## Section 1: Environment Setup & Data Loading

In [7]:
%pip install pandas scikit-learn xgboost shap imbalanced-learn openpyxl matplotlib seaborn numpy

  Using cached pandas-3.0.3-cp314-cp314-macosx_10_15_x86_64.whl.metadata (79 kB)
  Using cached scikit_learn-1.9.0-cp314-cp314-macosx_10_15_x86_64.whl.metadata (11 kB)
  Using cached xgboost-3.2.0-py3-none-macosx_10_15_x86_64.whl.metadata (2.1 kB)
  Using cached shap-0.52.0-cp312-abi3-macosx_10_13_x86_64.whl.metadata (26 kB)
  Using cached imbalanced_learn-0.14.2-py3-none-any.whl.metadata (8.9 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached matplotlib-3.10.9-cp314-cp314-macosx_10_13_x86_64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached numpy-2.4.6-cp314-cp314-macosx_10_15_x86_64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp314-cp314-macosx_10_14_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.22.1-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached tqdm-

In [6]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import openpyxl

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, matthews_corrcoef,
                             confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, roc_curve)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import shap

# Plot styling
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
PALETTE = ['#2E4057', '#1B6CA8', '#BDD5EA', '#F4A261', '#E76F51']

print("All libraries loaded successfully.")


ModuleNotFoundError: No module named 'numpy'

In [ ]:
# ── Load both sheets from the Excel file ─────────────────────────────────────
wb = openpyxl.load_workbook('Raw_Data.xlsx', read_only=True)

# --- Sociodemographic sheet ---
ws1 = wb['Sociodemographic Data']
rows1 = list(ws1.iter_rows(values_only=True))
header1 = rows1[1]
data1 = [r for r in rows1[2:] if r[0] is not None]
df_socio = pd.DataFrame(data1, columns=header1)
df_socio = df_socio.rename(columns={
    'No.': 'ID',
    'Smartphone-usage Duration': 'usage_duration',
    'Social Media Usage': 'social_media_usage',
    'The Most Frequent Access': 'frequent_access',
    'Gender': 'gender_socio',
    'Age': 'age',
    'Class': 'grade'
})
df_socio = df_socio.drop(columns=['Name/Initial'])

# --- Smartphone Addiction sheet ---
ws2 = wb['Smartphone Addiction']
rows2 = list(ws2.iter_rows(values_only=True))
data2 = []
for r in rows2[3:]:
    if r[0] is not None and isinstance(r[0], int):
        data2.append({
            'ID': r[0],
            'gender': r[1],
            'Q1': r[2], 'Q2': r[3], 'Q3': r[4], 'Q4': r[5],
            'Q5': r[6], 'Q6': r[7], 'Q7': r[8], 'Q8': r[9],
            'Q9': r[10], 'Q10': r[11],
            'sas_total': sum([x for x in [r[2],r[3],r[4],r[5],r[6],r[7],r[8],r[9],r[10],r[11]] if isinstance(x, (int,float))]),
            'addicted': r[13]
        })
df_add = pd.DataFrame(data2)

# --- Merge on ID ---
df = pd.merge(df_socio, df_add, on='ID')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


## Section 2: Exploratory Data Analysis (EDA) & Visualizations

In [ ]:
# ── 2.1 Dataset overview ─────────────────────────────────────────────────────
print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"Total records:        {len(df)}")
print(f"Features:             {df.shape[1]}")
print(f"Addicted (1):         {df['addicted'].sum()} ({df['addicted'].mean()*100:.1f}%)")
print(f"Not addicted (0):     {(df['addicted']==0).sum()} ({(df['addicted']==0).mean()*100:.1f}%)")
print(f"Female (P):           {(df['gender']=='P').sum()}")
print(f"Male (L):             {(df['gender']=='L').sum()}")
print(f"Age range:            {df['age'].min()} – {df['age'].max()}")
print(f"Missing values:       {df.isnull().sum().sum()}")
print("=" * 55)


In [ ]:
# ── 2.2 Class balance ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution bar chart
counts = df['addicted'].value_counts()
bars = axes[0].bar(['Not Addicted (0)', 'Addicted (1)'],
                   counts.values, color=[PALETTE[0], PALETTE[3]], width=0.5)
for bar, count in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 f'{count}\n({count/len(df)*100:.1f}%)',
                 ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Class Distribution (Addiction Label)', fontsize=14, fontweight='bold', pad=15)
axes[0].set_ylabel('Number of Participants')
axes[0].set_ylim(0, 300)

# Gender × Addiction stacked bar
gender_counts = df.groupby(['gender', 'addicted']).size().unstack(fill_value=0)
gender_counts.plot(kind='bar', ax=axes[1],
                   color=[PALETTE[0], PALETTE[3]], width=0.5)
axes[1].set_title('Addiction by Gender', fontsize=14, fontweight='bold', pad=15)
axes[1].set_ylabel('Count')
axes[1].set_xlabel('Gender (P=Female, L=Male)')
axes[1].legend(['Not Addicted', 'Addicted'])
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('fig_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: fig_class_balance.png")


In [ ]:
# ── 2.3 SAS-SV total score distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of SAS total scores
axes[0].hist(df[df['addicted']==0]['sas_total'], bins=20, alpha=0.7,
             color=PALETTE[0], label='Not Addicted', edgecolor='white')
axes[0].hist(df[df['addicted']==1]['sas_total'], bins=20, alpha=0.7,
             color=PALETTE[3], label='Addicted', edgecolor='white')
axes[0].axvline(x=31, color='red', linestyle='--', linewidth=1.5, label='Male cut-off (31)')
axes[0].axvline(x=33, color='darkred', linestyle=':', linewidth=1.5, label='Female cut-off (33)')
axes[0].set_title('SAS-SV Total Score Distribution', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('SAS-SV Total Score (range: 10–60)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Boxplot by addiction status
df_plot = df[['sas_total', 'addicted']].copy()
df_plot['Status'] = df_plot['addicted'].map({0: 'Not Addicted', 1: 'Addicted'})
groups = [df_plot[df_plot['Status']==s]['sas_total'].values for s in ['Not Addicted', 'Addicted']]
bp = axes[1].boxplot(groups, labels=['Not Addicted', 'Addicted'],
                     patch_artist=True,
                     boxprops=dict(facecolor=PALETTE[2]),
                     medianprops=dict(color=PALETTE[3], linewidth=2))
bp['boxes'][1].set_facecolor(PALETTE[3])
bp['boxes'][1].set_alpha(0.7)
axes[1].set_title('SAS-SV Score by Addiction Status', fontsize=14, fontweight='bold', pad=15)
axes[1].set_ylabel('SAS-SV Total Score')

plt.tight_layout()
plt.savefig('fig_sas_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 2.4 Individual SAS-SV item distributions ─────────────────────────────────
q_labels = [
    'Q1: Missing planned work',
    'Q2: Hard to concentrate',
    'Q3: Physical pain',
    'Q4: Cannot stand not having phone',
    'Q5: Impatient without phone',
    'Q6: Phone on mind when not using',
    'Q7: Will never give up phone',
    'Q8: Constantly checking social media',
    'Q9: Using longer than intended',
    'Q10: Others say I use too much'
]

q_cols = [f'Q{i}' for i in range(1, 11)]
q_means_add = df[df['addicted']==1][q_cols].mean()
q_means_not = df[df['addicted']==0][q_cols].mean()

fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(q_cols))
width = 0.35
bars1 = ax.bar(x - width/2, q_means_not.values, width, label='Not Addicted',
               color=PALETTE[0], alpha=0.85)
bars2 = ax.bar(x + width/2, q_means_add.values, width, label='Addicted',
               color=PALETTE[3], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f'Q{i}' for i in range(1, 11)])
ax.set_ylabel('Mean Score (1–6)')
ax.set_title('Mean SAS-SV Item Scores by Addiction Status', fontsize=14, fontweight='bold', pad=15)
ax.legend()
ax.set_ylim(0, 6.5)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('fig_item_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nItem descriptions:")
for i, label in enumerate(q_labels, 1):
    print(f"  Q{i}: {label.split(': ')[1]}")


In [ ]:
# ── 2.5 Smartphone usage duration distribution ───────────────────────────────
usage_map = {1: '1–3 hrs/day', 2: '4–6 hrs/day', 3: '7–9 hrs/day', 4: '9+ hrs/day'}
df['usage_label'] = df['usage_duration'].map(usage_map)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall usage duration
usage_counts = df['usage_label'].value_counts().reindex(usage_map.values())
axes[0].bar(usage_counts.index, usage_counts.values, color=PALETTE[:4], alpha=0.85)
axes[0].set_title('Daily Smartphone Usage Duration', fontsize=14, fontweight='bold', pad=15)
axes[0].set_ylabel('Number of Participants')
axes[0].tick_params(axis='x', rotation=15)

# Usage duration by addiction status
usage_add = df.groupby(['usage_label', 'addicted']).size().unstack(fill_value=0)
usage_add = usage_add.reindex(usage_map.values())
usage_add.plot(kind='bar', ax=axes[1], color=[PALETTE[0], PALETTE[3]], alpha=0.85)
axes[1].set_title('Usage Duration by Addiction Status', fontsize=14, fontweight='bold', pad=15)
axes[1].set_ylabel('Count')
axes[1].set_xlabel('')
axes[1].legend(['Not Addicted', 'Addicted'])
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('fig_usage_duration.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 2.6 Correlation heatmap ──────────────────────────────────────────────────
q_cols = [f'Q{i}' for i in range(1, 11)]
corr_cols = q_cols + ['sas_total', 'addicted', 'usage_duration', 'age']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('fig_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nTop correlations with addiction label:")
corr_with_target = corr_matrix['addicted'].drop('addicted').sort_values(ascending=False)
print(corr_with_target.to_string())


## Section 3: Data Preprocessing & Feature Engineering

In [ ]:
# ── 3.1 Encode gender ────────────────────────────────────────────────────────
df['gender_encoded'] = (df['gender'] == 'P').astype(int)  # 1=Female, 0=Male

# ── 3.2 Define feature matrix and target ─────────────────────────────────────
q_cols = [f'Q{i}' for i in range(1, 11)]
FEATURE_COLS = q_cols + ['sas_total', 'usage_duration', 'social_media_usage',
                          'frequent_access', 'age', 'gender_encoded']

X = df[FEATURE_COLS].copy()
y = df['addicted'].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape:         {y.shape}")
print(f"Features used:        {FEATURE_COLS}")


In [ ]:
# ── 3.3 Missing value check ──────────────────────────────────────────────────
missing = X.isnull().sum()
print("Missing values per feature:")
print(missing[missing > 0] if missing.sum() > 0 else "  None — dataset is complete.")


In [ ]:
# ── 3.4 Train / Validation / Test split (80 / 10 / 10) ──────────────────────
# Split off test set first — isolate before any modelling
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y)

# Split remaining into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.111, random_state=42, stratify=y_temp)
# 0.111 of 90% ≈ 10% of total

print(f"Training set:   {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.0f}%)")
print(f"Test set:       {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nClass balance in training set:")
print(y_train.value_counts().to_string())


In [ ]:
# ── 3.5 Class imbalance check and SMOTE ──────────────────────────────────────
ratio = y_train.value_counts()
imbalance_ratio = ratio.max() / ratio.min()
print(f"Class ratio in training set: {imbalance_ratio:.2f}:1")

if imbalance_ratio > 1.5:
    print(f"Imbalance detected ({imbalance_ratio:.2f}:1) — applying SMOTE...")
    smote = SMOTE(random_state=42)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
    print(f"After SMOTE — class distribution: {pd.Series(y_train_bal).value_counts().to_dict()}")
else:
    print("Class balance acceptable — SMOTE not applied.")
    X_train_bal, y_train_bal = X_train.copy(), y_train.copy()


## Section 4: Model Architecture & Training

### Three classifiers benchmarked:

| Model | Type | SHAP Support | Role |
|---|---|---|---|
| Logistic Regression | Linear baseline | Native | Establishes lower bound |
| Random Forest | Ensemble (bagging) | TreeSHAP | Strong baseline |
| XGBoost | Ensemble (gradient boosting) | TreeSHAP | Primary deployment candidate |

All models trained on the SMOTE-balanced training set and evaluated on the held-out test set.


In [ ]:
# ── 4.1 Define models ────────────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(
        C=1.0,
        max_iter=1000,
        solver='lbfgs',
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42,
        verbosity=0
    )
}

print("Models defined:")
for name, model in models.items():
    print(f"  {name}: {type(model).__name__}")


In [ ]:
# ── 4.2 Train all models and evaluate ────────────────────────────────────────
results = {}

for name, model in models.items():
    # Train
    model.fit(X_train_bal, y_train_bal)

    # Predict on test set
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Metrics
    results[name] = {
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1-Score':  round(f1_score(y_test, y_pred), 4),
        'AUC-ROC':   round(roc_auc_score(y_test, y_prob), 4),
        'MCC':       round(matthews_corrcoef(y_test, y_pred), 4),
        'model':     model,
        'y_pred':    y_pred,
        'y_prob':    y_prob
    }
    print(f"✓ {name} trained and evaluated.")

print("\nAll models complete.")


## Section 5: Performance Metrics & Model Comparison

In [ ]:
# ── 5.1 Metrics comparison table ─────────────────────────────────────────────
TARGET_ACCURACY = 0.85
TARGET_AUC = 0.87

metrics_df = pd.DataFrame({
    name: {k: v for k, v in scores.items() if k not in ['model', 'y_pred', 'y_prob']}
    for name, scores in results.items()
}).T

print("=" * 70)
print("MODEL PERFORMANCE COMPARISON (held-out test set)")
print("=" * 70)
print(metrics_df[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'MCC']].to_string())
print("=" * 70)
print(f"Target accuracy: ≥{TARGET_ACCURACY} | Target AUC: ≥{TARGET_AUC}")
print("=" * 70)

for name, scores in results.items():
    acc_ok  = "✓" if scores['Accuracy'] >= TARGET_ACCURACY else "✗"
    auc_ok  = "✓" if scores['AUC-ROC']  >= TARGET_AUC      else "✗"
    print(f"{name:25s}  Accuracy {acc_ok} ({scores['Accuracy']:.4f})  AUC {auc_ok} ({scores['AUC-ROC']:.4f})")


In [ ]:
# ── 5.2 Performance bar chart ────────────────────────────────────────────────
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
model_names  = list(results.keys())
x = np.arange(len(metric_names))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 6))
for i, (name, color) in enumerate(zip(model_names, PALETTE[:3])):
    vals = [results[name][m] for m in metric_names]
    bars = ax.bar(x + i*width, vals, width, label=name, color=color, alpha=0.88)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.axhline(y=TARGET_ACCURACY, color='red', linestyle='--', linewidth=1.5,
           label=f'Accuracy target ({TARGET_ACCURACY})')
ax.axhline(y=TARGET_AUC, color='darkred', linestyle=':', linewidth=1.5,
           label=f'AUC target ({TARGET_AUC})')
ax.set_xticks(x + width)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison (Test Set)', fontsize=14, fontweight='bold', pad=15)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('fig_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 5.3 Confusion matrices ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, scores) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, scores['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['Not Addicted', 'Addicted'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAcc={scores["Accuracy"]:.3f} | F1={scores["F1-Score"]:.3f}',
                 fontsize=11, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('fig_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 5.4 ROC curves ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
for name, color in zip(model_names, PALETTE[:3]):
    fpr, tpr, _ = roc_curve(y_test, results[name]['y_prob'])
    auc = results[name]['AUC-ROC']
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC = {auc:.3f})')
ax.plot([0,1],[0,1], 'k--', linewidth=1, label='Random baseline (AUC = 0.5)')
ax.axhline(y=0.87, color='gray', linestyle=':', linewidth=1, alpha=0.6, label='AUC target (0.87)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold', pad=15)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('fig_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 5.5 Select best model ────────────────────────────────────────────────────
best_name = max(
    {k: v for k, v in results.items()},
    key=lambda k: (results[k]['AUC-ROC'] + results[k]['F1-Score']) / 2
)
best_model  = results[best_name]['model']
best_scores = results[best_name]

print(f"Best model selected: {best_name}")
print(f"  Accuracy:  {best_scores['Accuracy']:.4f}")
print(f"  Precision: {best_scores['Precision']:.4f}")
print(f"  Recall:    {best_scores['Recall']:.4f}")
print(f"  F1-Score:  {best_scores['F1-Score']:.4f}")
print(f"  AUC-ROC:   {best_scores['AUC-ROC']:.4f}")
print(f"  MCC:       {best_scores['MCC']:.4f}")
print(f"\nThis model will be deployed in the FastAPI backend.")


In [ ]:
# ── 5.6 Fairness check by gender ─────────────────────────────────────────────
print("=" * 55)
print("FAIRNESS CHECK — Performance by Gender (Test Set)")
print("=" * 55)

X_test_copy = X_test.copy()
X_test_copy['y_true'] = y_test.values
X_test_copy['y_pred'] = best_scores['y_pred']
X_test_copy['gender_orig'] = df.loc[X_test.index, 'gender'].values

for gender_val, gender_label in [('P', 'Female'), ('L', 'Male')]:
    mask = X_test_copy['gender_orig'] == gender_val
    sub = X_test_copy[mask]
    if len(sub) < 5:
        print(f"{gender_label}: insufficient samples ({len(sub)})")
        continue
    acc = accuracy_score(sub['y_true'], sub['y_pred'])
    f1  = f1_score(sub['y_true'], sub['y_pred'], zero_division=0)
    print(f"{gender_label:8s} (n={len(sub):3d})  Accuracy={acc:.3f}  F1={f1:.3f}")

print("=" * 55)
print("Note: Disparities > 10pp will be documented as limitations.")


## Section 6: SHAP Explainability Analysis

SHAP (SHapley Additive exPlanations) uses game theory to assign each feature a contribution value for every individual prediction. TreeSHAP is used here for exact computation on tree-based models.

In the deployed system, the top 3 SHAP features per user are converted to plain-language template sentences shown on the results dashboard.


In [ ]:
# ── 6.1 Compute SHAP values ──────────────────────────────────────────────────
print("Computing SHAP values (this may take a moment)...")
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

# For binary classifiers, shap_values may be a list [class0, class1]
if isinstance(shap_values, list):
    shap_vals_pos = shap_values[1]  # class 1 = addicted
else:
    shap_vals_pos = shap_values

print(f"SHAP values computed for {shap_vals_pos.shape[0]} test samples.")
print(f"SHAP values shape: {shap_vals_pos.shape}")


In [ ]:
# ── 6.2 SHAP summary plot (beeswarm) ─────────────────────────────────────────
shap.summary_plot(shap_vals_pos, X_test,
                  feature_names=FEATURE_COLS,
                  plot_type='dot',
                  show=False)
plt.title('SHAP Summary Plot — Feature Impact on Addiction Prediction',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('fig_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 6.3 SHAP mean absolute importance bar chart ───────────────────────────────
mean_shap = np.abs(shap_vals_pos).mean(axis=0)
shap_importance = pd.Series(mean_shap, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = [PALETTE[3] if v > shap_importance.median() else PALETTE[0]
          for v in shap_importance.values]
bars = ax.barh(shap_importance.index, shap_importance.values, color=colors, alpha=0.88)
ax.set_xlabel('Mean |SHAP Value| (average impact on prediction)')
ax.set_title('Feature Importance (SHAP) — Global', fontsize=14, fontweight='bold', pad=15)
for bar, val in zip(bars, shap_importance.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
high_patch = mpatches.Patch(color=PALETTE[3], alpha=0.88, label='Above median importance')
low_patch  = mpatches.Patch(color=PALETTE[0], alpha=0.88, label='Below median importance')
ax.legend(handles=[high_patch, low_patch])
plt.tight_layout()
plt.savefig('fig_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nTop 5 most important features:")
print(shap_importance.tail(5).sort_values(ascending=False).to_string())


In [ ]:
# ── 6.4 SHAP template sentences (plain-language coaching output) ─────────────
# This is the lookup table deployed in the FastAPI backend.
# When the top 3 SHAP features are identified for a user,
# the corresponding sentences are returned to the frontend.

SHAP_TEMPLATES = {
    'Q1':                 "Missing planned work due to smartphone use is a key driver of your risk score.",
    'Q2':                 "Difficulty concentrating on academic work because of your phone is significantly increasing your risk.",
    'Q3':                 "Physical discomfort (wrist or neck pain) from phone use is contributing to your score.",
    'Q4':                 "Feeling unable to go without your smartphone is a strong indicator of dependency.",
    'Q5':                 "Feeling impatient or anxious when not holding your phone is a key risk factor.",
    'Q6':                 "Having your phone on your mind even when not using it is contributing to your risk.",
    'Q7':                 "Your sense that you would never give up your phone despite its impact is a significant risk indicator.",
    'Q8':                 "Constantly checking social media so as not to miss conversations is elevating your risk score.",
    'Q9':                 "Using your phone longer than you planned is one of the strongest contributors to your score.",
    'Q10':                "Others noticing and commenting on your phone use is a meaningful risk signal.",
    'sas_total':          "Your overall SAS-SV score is the primary driver of your addiction risk level.",
    'usage_duration':     "The total number of hours you spend on your phone each day is a key risk factor.",
    'social_media_usage': "Your social media usage patterns are contributing to your risk score.",
    'frequent_access':    "The type of content you access most frequently is influencing your risk level.",
    'age':                "Your age group is a contextual factor in your overall risk profile.",
    'gender_encoded':     "Your gender is a contextual factor considered in your risk assessment."
}

# Example: generate explanation for a single user
sample_idx = 0
sample_shap = shap_vals_pos[sample_idx]
top3_idx = np.argsort(np.abs(sample_shap))[::-1][:3]
top3_features = [FEATURE_COLS[i] for i in top3_idx]

print("=" * 60)
print("EXAMPLE USER — SHAP EXPLANATION OUTPUT")
print("=" * 60)
print(f"Risk score: {results[best_name]['y_prob'][sample_idx]*100:.0f}/100")
print(f"Predicted:  {'Addicted' if results[best_name]['y_pred'][sample_idx]==1 else 'Not Addicted'}")
print("\nTop 3 contributing factors:")
for i, feat in enumerate(top3_features, 1):
    shap_val = sample_shap[FEATURE_COLS.index(feat)]
    direction = "↑ increases" if shap_val > 0 else "↓ decreases"
    print(f"  {i}. [{feat}] {direction} risk by {abs(shap_val):.3f}")
    print(f"     → {SHAP_TEMPLATES[feat]}")


## Section 7: Summary & Next Steps

In [ ]:
# ── 7.1 Final summary ────────────────────────────────────────────────────────
print("=" * 65)
print("DIGITAL WELLBEING COACH — MODEL NOTEBOOK SUMMARY")
print("=" * 65)
print(f"\nDataset:         394 records | SAS-SV instrument | Indonesia")
print(f"Features:        {len(FEATURE_COLS)} features (10 SAS-SV items + sociodemographic)")
print(f"Train/Val/Test:  80% / 10% / 10%")
print(f"SMOTE:           Applied to training set (class imbalance correction)")
print(f"\nModels benchmarked:")
for name, scores in results.items():
    print(f"  {name:25s}  Acc={scores['Accuracy']:.3f}  F1={scores['F1-Score']:.3f}  AUC={scores['AUC-ROC']:.3f}")
print(f"\nBest model:      {best_name}")
print(f"\nAccuracy target (≥0.85):  {'MET ✓' if results[best_name]['Accuracy'] >= 0.85 else 'NOT MET ✗'}")
print(f"AUC target    (≥0.87):  {'MET ✓' if results[best_name]['AUC-ROC']  >= 0.87 else 'NOT MET ✗'}")
print(f"\nSHAP:           TreeSHAP computed | template sentences defined")
print("\nNext steps:")
print("  1. Deploy best model to FastAPI backend (Railway)")
print("  2. Connect React/Next.js frontend (Vercel)")
print("  3. Integrate SHAP template engine into /predict endpoint")
print("  4. Curate resource library (21 Kigali entries ready)")
print("  5. Pilot usability study with 30–50 Kigali students (post-REC approval)")
print("=" * 65)
